# Context Processor
> Utilities for expanding, compacting, and canonicalising JSON-LD using **pyld** while delegating all network fetches to our registry-aware, cache-memoised loader.

In [ ]:
#| default_exp core.context

## imports  


In [ ]:
#| export
from __future__ import annotations

import json, hashlib
from functools import lru_cache
from typing import Any, Dict, List

from pyld import jsonld

from cogitarelink.core.debug import get_logger
from cogitarelink.core.cache import InMemoryCache
from cogitarelink.vocab.registry import registry
from cogitarelink.vocab.composer import composer

log = get_logger("context")
_cache = InMemoryCache(maxsize=512)

##  registry-aware document loader

In [ ]:
#| export
@_cache.memoize("jsonld-doc")
def _http_get(url: str) -> Dict[str, Any]:
    import httpx, json as _json
    log.debug(f"doc-loader GET {url}")
    r = httpx.get(url, follow_redirects=True, timeout=10)
    r.raise_for_status()
    return dict(contextUrl=None, documentUrl=url, document=r.text)

def get_document_loader():
    "Return a pyld-compatible loader that understands registry *prefixes and URIs*."
    _mem: Dict[str, Dict[str, Any]] = {}

    def loader(url: str):
        # 1️⃣  Try registry (prefix **or** alias URL)
        try:
            entry = registry.resolve(url)
            ctx   = entry.context_payload()
            return dict(contextUrl=None, documentUrl=url, document=json.dumps(ctx))
        except KeyError:
            pass  # not a known vocab – proceed

        # 2️⃣  Memoised HTTP fetch (sandboxed envs may still block; caller must catch)
        if url not in _mem:
            _mem[url] = _http_get(url)
        return _mem[url]

    return loader

# Register globally so every pyld call uses it
jsonld.set_document_loader(get_document_loader())

## ContextProcessor  

In [ ]:
#| export
class ContextProcessor:
    "Thin wrapper around pyld with sensible defaults."

    def expand(self, doc: Dict[str, Any] | List[Any]) -> List[Dict[str, Any]]:
        "Return expanded JSON-LD array."
        return jsonld.expand(doc, options=dict(base=None))

    def compact(self, doc: Dict[str, Any] | List[Any],
                ctx: Dict[str, Any] | List[Any]) -> Dict[str, Any]:
        "Compact against a given `@context`."
        return jsonld.compact(doc, ctx, options=dict(base=None))

    def normalize(self, doc: Dict[str, Any] | List[Any]) -> str:
        """
        Canonicalise using URDNA2015 and return **N-Quads** string.
        Useful for later signing.
        """
        return jsonld.normalize(
            doc,
            options=dict(algorithm="URDNA2015", format="application/n-quads")
        )

    # convenience ------------------------------------------------------
    def expand_with(self, prefixes: List[str], doc: Dict[str, Any]) -> List[Dict[str, Any]]:
        "Inject composed context, then expand."
        ctx = composer.compose(prefixes)
        merged = dict(doc, **ctx)
        return self.expand(merged)

## quick tests  

In [ ]:
#| hide
from fastcore.test import test_eq
import pytest

proc = ContextProcessor()

# 4.1 basic expand
example = {"@context": {"name": "http://schema.org/name"}, "name": "Ada"}
exp = proc.expand(example)
assert isinstance(exp, list), "Expected expanded result to be a list"
test_eq(exp[0]["http://schema.org/name"][0]["@value"], "Ada")

# 4.2 cache verifies: second call should hit memo
url = "https://schema.org/docs/jsonldcontext.jsonld"
_first  = get_document_loader()(url)        # fetch & cache
_second = get_document_loader()(url)        # cached
test_eq(_first["document"], _second["document"])

In [ ]:
#| hide
# 1. Test compact method
compact_example = {
    "@id": "http://example.org/person",
    "http://schema.org/name": [{"@value": "Ada Lovelace"}]
}
context = {"@context": {"name": "http://schema.org/name"}}
compacted = proc.compact(compact_example, context)
test_eq(compacted["name"], "Ada Lovelace")

# 2. Test normalize method
norm_example = {"@context": {"name": "http://schema.org/name"}, "name": "Ada"}
normalized = proc.normalize(norm_example)
assert isinstance(normalized, str), "Expected normalized result to be a string"
assert "http://schema.org/name" in normalized, "Expected normalized result to contain expanded predicate"


In [ ]:
#| hide
# 3. Test expand_with method
# Assuming your registry has some predefined prefixes like 'schema'
simple_doc = {"name": "Ada Lovelace"}
expanded = proc.expand_with(["schema"], simple_doc)

# Check that expansion happened with the schema context
assert isinstance(expanded, list), "Expected expanded result to be a list"
assert "http://schema.org/name" in expanded[0], "Expected expanded document to use schema.org context"


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()